<a href="https://colab.research.google.com/github/ASUcicilab/GIS322/blob/main/notebook/Module_5_(3)_Geocoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.3 Geocoding

<br></br>
What is geocoding?

<br></br>

Geocoding is to converti addresses into coordinates (or vice versa), which is a really common GIS task. For example, we might want to know the longitude and latitude from a business address. Fortunately, there are convinient libraries in python that make the geocoding task really easy.



<br></br>
One of the libraries that can do the geocoding for us is `geopy` (https://geopy.readthedocs.io/en/stable/#module-geopy.geocoders) that makes it easy to locate the coordinates of addresses, cities, countries, and landmarks across the globe using third-party geocoders and other data sources. You can think about `geopy` as a "wrapper"; however, what's inside is geocoder services, and many of them are not FREE! (also can be very expensive if you have large queries). Some of the providers are:

* ArcGIS online geocoding service (https://geocode.arcgis.com).
* Google Geocoding API (https://developers.google.com/maps/documentation/geocoding/start).
* OpenStreetMap (Nominatim) (https://nominatim.openstreetmap.org).




<br></br>
Great thing about `geopy` is that it is integrated in `geopandas`, which makes it very friendly (native) to use. Now let's give it a try.





In [1]:
my_address = "1037 South Farmer Ave, Tempe, Arizona"

<br></br>
Above is my apartment address, is someone living close to me?
<br></br>
And next, let's try to convert this address to coordinates. The command is:


<br></br>
`geopandas.tools.geocode.eocode(my_address, provider='nominatim')`
    
where we use the OpenStreetMap's Nominatim's geocoding service.


In [2]:
!pip install geopandas
!pip install geopandas geopy
from geopandas.tools import geocode
# Geocode addresses with Nominatim backend
coords = geocode(my_address, provider='nominatim', user_agent='autogis_xx')

In [3]:
coords

,geometry,address
0,POINT (-111.94392 33.41761),"1037, South Farmer Avenue, Tempe Junction, Tem..."




<br></br>
What we get is a GeoDataFrame with only one row that contains my address and the coordinates, which seems to be quite accurate. (you can validate it on Google Maps)

<br></br>
`POINT (-111.944135926829 33.4176156097561)`



<br></br>
What if we have multiple addresses? We just need to give the `geocode` function a list of addresses.
<br></br>
For example, if we have two locations: My home and Desert Financial Arena.



In [4]:
my_address_list = ["1037 South Farmer Ave, Tempe, Arizona",
                   "Desert Financial Arena, Tempe, Arizona"]

coords_list = geocode(my_address_list, provider='nominatim', user_agent='autogis_xx')

In [5]:
coords_list

,geometry,address
0,POINT (-111.94392 33.41761),"1037, South Farmer Avenue, Tempe Junction, Tem..."
1,POINT (-111.93101 33.42451),"Desert Financial Arena, 600, East Veterans Way..."


That being said, if we have a Excel table of addresses and needed be geocoded, we can do this in a batch.

Let's try to plot those two locations on a map using the coordinates we obtained from the geocoder (I will talk about how to make a map like below  in more details in the Module 6).

In [6]:
import folium
%matplotlib inline
asu = folium.Map(
    location=[33.4200, -111.9350],
    zoom_start=15
)
folium.Marker([coords_list.geometry[1].y, coords_list.geometry[1].x], popup='<i>Arena</i>').add_to(asu)
folium.Marker([coords_list.geometry[0].y, coords_list.geometry[0].x], popup='<i>Home</i>').add_to(asu)
asu

<br></br>
## Geocoding many addresses

Two addresses work fine. But what happens when we have a whole table of addresses, like 20, or 500 Starbucks stores?

Below is a table of 20 places around the Phoenix metro area.

In [7]:
import pandas as pd

address_df = pd.DataFrame({
    "name": ["My apartment", "Desert Financial Arena", "Mountain America Stadium",
             "Tempe Center for the Arts", "Tempe Marketplace", "Arizona Mills",
             "Arizona State Capitol", "Phoenix Sky Harbor Airport", "Chase Field",
             "Footprint Center", "Arizona Science Center", "Phoenix Art Museum",
             "Heard Museum", "Desert Botanical Garden", "Phoenix Zoo",
             "Musical Instrument Museum", "Scottsdale Fashion Square", "Taliesin West",
             "Mesa Arts Center", "State Farm Stadium"],
    "address": ["1037 South Farmer Ave, Tempe, AZ",
                "600 E Veterans Way, Tempe, AZ 85281",
                "500 E Veterans Way, Tempe, AZ 85281",
                "700 W Rio Salado Pkwy, Tempe, AZ 85281",
                "2000 E Rio Salado Pkwy, Tempe, AZ 85281",
                "5000 S Arizona Mills Cir, Tempe, AZ 85282",
                "1700 W Washington St, Phoenix, AZ 85007",
                "3400 E Sky Harbor Blvd, Phoenix, AZ 85034",
                "401 E Jefferson St, Phoenix, AZ 85004",
                "201 E Jefferson St, Phoenix, AZ 85004",
                "600 E Washington St, Phoenix, AZ 85004",
                "1625 N Central Ave, Phoenix, AZ 85004",
                "2301 N Central Ave, Phoenix, AZ 85004",
                "1201 N Galvin Pkwy, Phoenix, AZ 85008",
                "455 N Galvin Pkwy, Phoenix, AZ 85008",
                "4725 E Mayo Blvd, Phoenix, AZ 85050",
                "7014 E Camelback Rd, Scottsdale, AZ 85251",
                "12621 N Frank Lloyd Wright Blvd, Scottsdale, AZ 85259",
                "1 E Main St, Mesa, AZ 85201",
                "1 Cardinals Dr, Glendale, AZ 85305"],
})

address_df

,name,address
0,My apartment,"1037 South Farmer Ave, Tempe, AZ"
1,Desert Financial Arena,"600 E Veterans Way, Tempe, AZ 85281"
2,Mountain America Stadium,"500 E Veterans Way, Tempe, AZ 85281"
3,Tempe Center for the Arts,"700 W Rio Salado Pkwy, Tempe, AZ 85281"
4,Tempe Marketplace,"2000 E Rio Salado Pkwy, Tempe, AZ 85281"
5,Arizona Mills,"5000 S Arizona Mills Cir, Tempe, AZ 85282"
6,Arizona State Capitol,"1700 W Washington St, Phoenix, AZ 85007"
7,Phoenix Sky Harbor Airport,"3400 E Sky Harbor Blvd, Phoenix, AZ 85034"
8,Chase Field,"401 E Jefferson St, Phoenix, AZ 85004"
9,Footprint Center,"201 E Jefferson St, Phoenix, AZ 85004"


<br></br>
### The problem: sending too many requests too fast

Free geocoding services only allow about **1 request per second**. `geopandas.tools.geocode` sends the addresses one after another, and for most providers (e.g. Photon) it does **not wait** between them. With a long list, the server starts refusing requests and the whole cell stops with an error like:

```
AdapterHTTPError: Non-successful status code 503
GeocoderTimedOut: Non-successful status code 503
```

`503 Service Unavailable` means "too many requests, slow down". One refused address is enough to lose the results for **all** addresses.

### The solution: `RateLimiter`

`geopy` has a `RateLimiter` that wraps the geocoder and:

* waits `min_delay_seconds` between requests (stay under the limit),
* retries a failed request up to `max_retries` times, waiting `error_wait_seconds` first (back off after a 503),
* with `swallow_exceptions=True`, gives `None` for an address that still fails, instead of stopping everything.

Then we build the GeoDataFrame ourselves from the results.

In [8]:
import geopandas as gpd
from geopy.geocoders import Photon
from geopy.extra.rate_limiter import RateLimiter
from shapely.geometry import Point

geolocator = Photon(user_agent="autogis_xx", timeout=10)   # make user_agent unique
geocode_rl = RateLimiter(geolocator.geocode,
                         min_delay_seconds=1,     # slow down
                         max_retries=3,
                         error_wait_seconds=5,    # back off after a 503
                         swallow_exceptions=True) # failed rows become None

locs = address_df["address"].apply(geocode_rl)

address_gdf = gpd.GeoDataFrame(
    address_df.copy(),
    geometry=[Point(l.longitude, l.latitude) if l else None for l in locs],
    crs="EPSG:4326",
)
address_gdf["geocoded_address"] = [l.address if l else None for l in locs]

print(address_gdf.geometry.isna().sum(), "of", len(address_gdf), "failed")
address_gdf

0 of 20 failed


,name,address,geometry,geocoded_address
0,My apartment,"1037 South Farmer Ave, Tempe, AZ",POINT (-111.94392 33.41761),"1037, South Farmer Avenue, 85280, South Farmer..."
1,Desert Financial Arena,"600 E Veterans Way, Tempe, AZ 85281",POINT (-111.93101 33.42451),"Desert Financial Arena, 600, East Veterans Way..."
2,Mountain America Stadium,"500 E Veterans Way, Tempe, AZ 85281",POINT (-111.93266 33.42631),"Mountain America Stadium, 500, East Veterans W..."
3,Tempe Center for the Arts,"700 W Rio Salado Pkwy, Tempe, AZ 85281",POINT (-111.94921 33.43096),"Rio Salado Pkwy & 700 West, West Rio Salado Pa..."
4,Tempe Marketplace,"2000 E Rio Salado Pkwy, Tempe, AZ 85281",POINT (-111.90105 33.42944),"Rio Salado Pkwy & Smith Rd, 2000, East Rio Sal..."
5,Arizona Mills,"5000 S Arizona Mills Cir, Tempe, AZ 85282",POINT (-111.96459 33.38317),"Arizona Mills, 5000, South Arizona Mills Circl..."
6,Arizona State Capitol,"1700 W Washington St, Phoenix, AZ 85007",POINT (-112.09662 33.44863),"Arizona State House of Representatives, 1700, ..."
7,Phoenix Sky Harbor Airport,"3400 E Sky Harbor Blvd, Phoenix, AZ 85034",POINT (-112.00679 33.43285),"Phoenix Sky Harbor International Airport, Copp..."
8,Chase Field,"401 E Jefferson St, Phoenix, AZ 85004",POINT (-112.06791 33.44616),"401, East Jefferson Street, 85004, East Jeffer..."
9,Footprint Center,"201 E Jefferson St, Phoenix, AZ 85004",POINT (-112.07451 33.44711),"Downtown Phoenix Hub/Jefferson Street, West Je..."


<br></br>
This takes about 1 second per address (20 addresses ≈ 20 seconds, 500 addresses ≈ 8–9 minutes), but it finishes without errors. Any address that failed has an empty geometry, so drop those rows before mapping:

In [9]:
address_gdf = address_gdf[address_gdf.geometry.notna()]

phx = folium.Map(location=[33.48, -111.95], zoom_start=10)
for _, row in address_gdf.iterrows():
    folium.Marker([row.geometry.y, row.geometry.x], popup=row["name"]).add_to(phx)
phx